In [0]:
base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/" 


yellowSchema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
    #,
    #StructField("cbd_congestion_fee", DoubleType(), True)
])


yellow_taxi_raw_df = (
    spark.read
    .format("parquet")
    .schema(yellowSchema)
    .load(yellow_path)
)

In [0]:
%pip install h3 

In [0]:
%pip install geopandas

In [0]:
%pip install folium

In [0]:
import h3
import geopandas as gpd
import folium
from shapely.geometry import mapping

# 1. 加载并转换坐标系 (确保列名匹配你的 Index: zone, LocationID, borough)
gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp").to_crs("EPSG:4326")

# 2. 定义 H3 转换函数 (使用你之前调通的质心方案，最稳健)
def get_h3_cell(geom, res=8):
    centroid = geom.centroid
    try:
        # 兼容 H3 4.x 和 3.x
        return h3.latlng_to_cell(centroid.y, centroid.x, res) if hasattr(h3, 'latlng_to_cell') else h3.geo_to_h3(centroid.y, centroid.x, res)
    except:
        return None

# 3. 生成数据
gdf['centroid_h3_cells'] = gdf['geometry'].apply(get_h3_cell)
# 这一步定义变量 h3_df_exploded
h3_df_exploded = gdf[['LocationID', 'borough', 'zone', 'centroid_h3_cells']].copy()


In [0]:
# 1. 将 Pandas DataFrame 转换为 Spark DataFrame
# 注意：转换时 Spark 会根据 Pandas 的数据自动推断 Schema
from pyspark.sql.window import Window 
from pyspark.sql.functions import row_number, col 

windowSpec = Window.partitionBy("VendorID","tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy(col("tolls_amount").desc())

# 2. 增加行号并过滤
yellow_taxi_df = (
    yellow_taxi_raw_df
    .withColumn("row_num", row_number().over(windowSpec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

spark_h3_df = spark.createDataFrame(h3_df_exploded)


# 2. 执行 Join (现在两边都是 Spark DataFrame 了)
yellow_taxi_enriched_trips = yellow_taxi_df.join(
    spark_h3_df.select("LocationID", "centroid_h3_cells"), 
    on=yellow_taxi_df.PULocationID == spark_h3_df.LocationID, 
    how="inner"
)

# 3. 检查结果
# 注意：请确认字段名是 lpep_... 还是 tpep_...（根据你之前定义的 Schema 应该是 lpep）
yellow_taxi_enriched_trips.select("tpep_pickup_datetime", "PULocationID", "centroid_h3_cells").show(10)

In [0]:
from pyspark.sql import functions as F

# 1. 准备基础分析表
# 我们使用你之前 Join 好的 distance_analysis
hotspot_base_df = distance_analysis.select(
    "h3_cells", 
    "pickupDatetime", 
    "passengerID", # 如果没有则用 count(*)
    "vendorID"
).withColumn("pickup_hour", F.hour("pickupDatetime")) \
 .withColumn("day_of_week", F.dayofweek("pickupDatetime")) \
 .withColumn("day_name", F.date_format("pickupDatetime", "EEEE")) # 转化为 Monday, Tuesday 等

# 2. 检查一下转换后的数据
hotspot_base_df.select("pickupDatetime", "pickup_hour", "day_name").show(5)

### 定义核心指标：每小时净收入 (Net Hourly Earnings)
在 2016 年，我们要衡量一个 H3 格子的“含金量”，不能只看总收入，要看时间产出比。

我们将计算：

Average Fare per Trip: 每一单平均能收多少钱。

Efficiency Index: 每分钟行程能赚多少钱（Fare / Trip Duration）。

In [0]:
from pyspark.sql import functions as F

# 1. 预处理：计算行程时长（分钟）
# 剔除时长为 0 或负数的异常数据
efficiency_base = yellow_taxi_enriched_trips.withColumn(
    "duration_min", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
).filter("duration_min > 0 and duration_min < 180") # 过滤掉超过3小时的极端异常值

# 2. 按 H3 和 小时 聚合
revenue_stats = efficiency_base.groupBy("h3_cells", "pickup_hour") \
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg(F.col("fare_amount") / F.col("duration_min")).alias("earnings_per_min")
    )

### 寻找“黄金格子” (The Golden Hexagons)
我们要找出那些订单量不低，且每分钟收益最高的区域。

In [0]:
# 过滤掉订单量太少的格子（比如一小时少于 10 单的），避免统计偏差
golden_grids = revenue_stats.filter("trip_count > 10") \
    .orderBy(F.desc("earnings_per_min"))

print("2016年 赚钱效率最高的 Top 10 H3 格子：")
golden_grids.show(10)